# DB-Rechnung & Break-Even-Analyse – KI im Controlling

**Ziel:** Python-basierte Analyse der DB-Rechnung und Break-Even-Simulation  
**Datenquelle:** `DB_Rechnung_5J_Simulation_break_even.xlsx` (im gleichen Ordner)  
**Anwendungsfall:** Schulungsübung – Modul 3: Claude in Excel  

---

**Wie dieser Code entstanden ist:**  
Dieser Code wurde von Claude (claude.ai) generiert auf Basis der Schulungsübung:  
> *"Berechne den Break-Even-Umsatz, erstelle eine Sensitivitätstabelle und Szenarioanalyse..."*

**Abhängigkeiten:** `pandas`, `openpyxl`, `plotly`, `numpy`  
Installation (falls nicht vorhanden):
```
pip install pandas openpyxl plotly numpy
```

---

> ⚠️ **Hinweis:** Alle Berechnungen auf Basis der Schulungs-Demodaten – keine realen Unternehmensdaten verwenden!

In [ ]:
# =============================================================================
# IMPORTS
# Standard Library
import warnings
warnings.filterwarnings('ignore')

# Third Party
import pandas as pd                     # Data manipulation & Excel reading
import numpy as np                      # Numerical operations
import plotly.graph_objects as go       # Interactive charts
import plotly.express as px             # Quick charts
from plotly.subplots import make_subplots  # Multi-panel charts
from IPython.display import display, HTML  # Notebook output

print('Libraries loaded successfully.')

In [ ]:
# =============================================================================
# CONFIGURATION
# Central parameters – adjust here, not scattered in the code below

FILE_PATH = 'DB_Rechnung_5J_Simulation_break_even.xlsx'  # Excel file path

# Break-Even scenario parameters (used if manual override needed)
FIXED_COST_INCREASE_SCENARIOS = [0.05, 0.10, 0.15]  # +5%, +10%, +15%

# Scenario analysis parameters
REVENUE_DELTA    = 0.10   # ±10% revenue variation
MATERIAL_DELTA   = 0.05   # ±5%  material cost variation

# Chart colors
COLOR_BASE  = '#2563ab'   # Blue
COLOR_BEST  = '#16a34a'   # Green
COLOR_WORST = '#dc2626'   # Red
COLOR_GRID  = '#dde3ef'   # Light blue-grey

## Schritt 1 – Excel-Datei laden und Struktur analysieren

In [ ]:
# =============================================================================
# DATA LOADING – Excel file inspection

def load_excel_file(filepath: str) -> dict:
    """
    Load all sheets from an Excel file and return as a dict of DataFrames.
    Prints available sheets for user orientation.

    Args:
        filepath: Path to the .xlsx file
    Returns:
        Dict {sheet_name: DataFrame}
    Raises:
        FileNotFoundError if file not found
    """
    try:
        xls = pd.ExcelFile(filepath)
        print(f'Datei geladen: {filepath}')
        print(f'Verfügbare Tabellenblätter ({len(xls.sheet_names)}):')
        for i, sheet in enumerate(xls.sheet_names, 1):
            print(f'  {i}. {sheet}')
        sheets = {sheet: pd.read_excel(xls, sheet_name=sheet) for sheet in xls.sheet_names}
        return sheets
    except FileNotFoundError:
        raise FileNotFoundError(
            f'Datei nicht gefunden: {filepath}\n'
            'Bitte sicherstellen, dass die Datei im gleichen Ordner wie dieses Notebook liegt.'
        )


sheets = load_excel_file(FILE_PATH)

# Show first sheet preview
first_sheet_name = list(sheets.keys())[0]
print(f'\nVorschau Tabellenblatt "{first_sheet_name}":')
display(sheets[first_sheet_name].head(20))

In [ ]:
# =============================================================================
# SHEET EXPLORATION
# Show dimensions and column structure for each sheet

for name, df in sheets.items():
    print(f'\n── Sheet: "{name}" ──')
    print(f'   Zeilen: {df.shape[0]} | Spalten: {df.shape[1]}')
    print(f'   Spalten: {list(df.columns)}')

## Schritt 2 – Break-Even-Analyse

**Formel:** Break-Even-Umsatz = Fixkosten / (1 − variable Kostenquote)  
→ Passe die Variablen unten an die tatsächlichen Spaltennamen aus Schritt 1 an.

In [ ]:
# =============================================================================
# BREAK-EVEN CALCULATION
#
# ANPASSEN: Trage hier die tatsächlichen Werte aus der Excel-Datei ein
# (entweder manuell aus dem Sheet lesen oder über Pandas extrahieren)
#
# Beispielstruktur – ersetze mit echten Werten aus sheets[first_sheet_name]

def calculate_break_even(
    revenue: float,
    variable_costs: float,
    fixed_costs: float,
    label: str = 'Basis'
) -> dict:
    """
    Calculate break-even revenue, safety margin, and contribution margin ratio.

    Args:
        revenue: Total revenue in €
        variable_costs: Total variable costs in €
        fixed_costs: Total fixed costs in €
        label: Scenario label for display
    Returns:
        Dict with all break-even metrics
    """
    if revenue <= 0:
        raise ValueError('Revenue must be > 0')

    cm1 = revenue - variable_costs                    # Deckungsbeitrag I (absolut)
    cm1_ratio = cm1 / revenue                         # CM1-Quote

    if cm1_ratio <= 0:
        raise ValueError('CM1-Quote <= 0: keine Gewinnschwelle erreichbar')

    break_even_revenue = fixed_costs / cm1_ratio      # Gewinnschwellen-Umsatz
    safety_margin_abs  = revenue - break_even_revenue # Sicherheitsspanne absolut
    safety_margin_pct  = safety_margin_abs / revenue  # Sicherheitsspanne relativ
    ebitda             = cm1 - fixed_costs            # EBITDA (ohne Sonstige)
    ebitda_margin      = ebitda / revenue             # EBITDA-Marge

    return {
        'Szenario':                  label,
        'Umsatz (€)':                round(revenue, 0),
        'Variable Kosten (€)':       round(variable_costs, 0),
        'CM1 absolut (€)':           round(cm1, 0),
        'CM1-Quote (%)':             round(cm1_ratio * 100, 1),
        'Fixkosten (€)':             round(fixed_costs, 0),
        'Break-Even-Umsatz (€)':     round(break_even_revenue, 0),
        'Sicherheitsspanne (€)':     round(safety_margin_abs, 0),
        'Sicherheitsspanne (%)':     round(safety_margin_pct * 100, 1),
        'EBITDA (€)':                round(ebitda, 0),
        'EBITDA-Marge (%)':          round(ebitda_margin * 100, 1),
    }


# ─── HIER ANPASSEN: Werte aus Excel-Datei eintragen ───────────────────────────
# Entweder manuell (nach Inspektion in Schritt 1) oder per Pandas-Lookup:
#   df = sheets['Tabellenblattname']
#   revenue = df.loc[df['Bezeichnung'] == 'Umsatz', 'Jahr1'].values[0]

REVENUE        = 100_000_000   # Umsatz in € – BITTE ERSETZEN
VARIABLE_COSTS =  55_000_000   # Variable Kosten in € – BITTE ERSETZEN
FIXED_COSTS    =  30_000_000   # Fixkosten in € – BITTE ERSETZEN
# ──────────────────────────────────────────────────────────────────────────────

base_result = calculate_break_even(REVENUE, VARIABLE_COSTS, FIXED_COSTS, 'Basis (Ist)')

print('\n=== BREAK-EVEN-ANALYSE ===')
for key, val in base_result.items():
    print(f'  {key:<30}: {val:>15,}' if isinstance(val, (int, float)) else f'  {key:<30}: {val}')

## Schritt 3 – Sensitivitätstabelle: Fixkostenvariation

In [ ]:
# =============================================================================
# SENSITIVITY TABLE – Fixed cost variations

def fixed_cost_sensitivity(
    revenue: float,
    variable_costs: float,
    base_fixed_costs: float,
    deltas: list
) -> pd.DataFrame:
    """
    Calculate break-even for different fixed cost levels.

    Args:
        revenue: Base revenue
        variable_costs: Base variable costs
        base_fixed_costs: Baseline fixed costs
        deltas: List of % changes (e.g. [0.05, 0.10, 0.15])
    Returns:
        DataFrame with sensitivity results
    """
    rows = []
    # Add baseline (0% change)
    for delta in [0.0] + deltas:
        fc_new = base_fixed_costs * (1 + delta)
        result = calculate_break_even(revenue, variable_costs, fc_new,
                                      label=f'Fixkosten {delta:+.0%}')
        rows.append(result)
    return pd.DataFrame(rows)


sensitivity_df = fixed_cost_sensitivity(
    REVENUE, VARIABLE_COSTS, FIXED_COSTS,
    FIXED_COST_INCREASE_SCENARIOS
)

print('\n=== SENSITIVITÄTSTABELLE: Fixkostenvariation ===')
display(sensitivity_df[[
    'Szenario', 'Fixkosten (€)', 'Break-Even-Umsatz (€)',
    'Sicherheitsspanne (€)', 'Sicherheitsspanne (%)', 'EBITDA (€)'
]])

## Schritt 4 – Szenarioanalyse: Base / Best / Worst

In [ ]:
# =============================================================================
# SCENARIO ANALYSIS – Base / Best / Worst

scenarios = {
    'Base':  {'revenue': REVENUE,
               'variable_costs': VARIABLE_COSTS,
               'fixed_costs':    FIXED_COSTS},
    'Best':  {'revenue':         REVENUE * (1 + REVENUE_DELTA),
               'variable_costs': VARIABLE_COSTS * (1 - MATERIAL_DELTA),
               'fixed_costs':    FIXED_COSTS},
    'Worst': {'revenue':         REVENUE * (1 - REVENUE_DELTA),
               'variable_costs': VARIABLE_COSTS * (1 + MATERIAL_DELTA),
               'fixed_costs':    FIXED_COSTS},
}

scenario_results = []
for name, params in scenarios.items():
    result = calculate_break_even(
        params['revenue'], params['variable_costs'], params['fixed_costs'],
        label=name
    )
    scenario_results.append(result)

scenario_df = pd.DataFrame(scenario_results)
print('\n=== SZENARIOANALYSE: Base / Best / Worst ===')
display(scenario_df[[
    'Szenario', 'Umsatz (€)', 'CM1-Quote (%)',
    'Break-Even-Umsatz (€)', 'Sicherheitsspanne (%)',
    'EBITDA (€)', 'EBITDA-Marge (%)'
]])

## Schritt 5 – Visualisierung

In [ ]:
# =============================================================================
# VISUALIZATION 1 – Waterfall: Break-Even Bridge

fig_waterfall = go.Figure(go.Waterfall(
    name='Break-Even Bridge',
    orientation='v',
    measure=['absolute', 'relative', 'total', 'relative', 'total'],
    x=['Umsatz', 'Variable Kosten', 'CM1', 'Fixkosten', 'EBITDA'],
    y=[
        REVENUE,
        -VARIABLE_COSTS,
        0,                         # total
        -FIXED_COSTS,
        0                          # total
    ],
    connector={'line': {'color': COLOR_GRID}},
    increasing={'marker': {'color': COLOR_BEST}},
    decreasing={'marker': {'color': COLOR_WORST}},
    totals={'marker': {'color': COLOR_BASE}},
    text=[
        f'{REVENUE/1e6:.1f} Mio. €',
        f'-{VARIABLE_COSTS/1e6:.1f} Mio. €',
        f'{(REVENUE-VARIABLE_COSTS)/1e6:.1f} Mio. €',
        f'-{FIXED_COSTS/1e6:.1f} Mio. €',
        f'{(REVENUE-VARIABLE_COSTS-FIXED_COSTS)/1e6:.1f} Mio. €'
    ],
    textposition='outside'
))

fig_waterfall.update_layout(
    title='P&L-Brücke: Umsatz → CM1 → EBITDA (Basis-Szenario)',
    yaxis_title='€',
    template='plotly_white',
    height=450,
    showlegend=False
)
fig_waterfall.show()
fig_waterfall.write_html('DB_Rechnung_Waterfall.html')
print('Waterfall gespeichert als DB_Rechnung_Waterfall.html')

In [ ]:
# =============================================================================
# VISUALIZATION 2 – Scenario Comparison: EBITDA & Break-Even

fig_scenarios = make_subplots(
    rows=1, cols=2,
    subplot_titles=['EBITDA je Szenario (€)', 'Break-Even-Umsatz je Szenario (€)']
)

sc_names  = [r['Szenario'] for r in scenario_results]
sc_ebitda = [r['EBITDA (€)'] for r in scenario_results]
sc_be     = [r['Break-Even-Umsatz (€)'] for r in scenario_results]
colors    = [COLOR_BASE, COLOR_BEST, COLOR_WORST]

fig_scenarios.add_trace(
    go.Bar(x=sc_names, y=sc_ebitda, marker_color=colors,
           text=[f'{v/1e6:.1f} Mio. €' for v in sc_ebitda], textposition='outside'),
    row=1, col=1
)
fig_scenarios.add_trace(
    go.Bar(x=sc_names, y=sc_be, marker_color=colors,
           text=[f'{v/1e6:.1f} Mio. €' for v in sc_be], textposition='outside'),
    row=1, col=2
)

fig_scenarios.update_layout(
    title='Szenarioanalyse – Base / Best / Worst',
    template='plotly_white',
    height=420,
    showlegend=False
)
fig_scenarios.show()
fig_scenarios.write_html('DB_Rechnung_Szenarien.html')
print('Szenario-Chart gespeichert als DB_Rechnung_Szenarien.html')

## Schritt 6 – Executive Summary

In [ ]:
# =============================================================================
# EXECUTIVE SUMMARY

def generate_executive_summary(base: dict, scenarios: list) -> str:
    """
    Generate a plain-text CFO-ready executive summary.

    Args:
        base: Base scenario result dict
        scenarios: List of all scenario result dicts
    Returns:
        Formatted summary string
    """
    lines = ['=' * 65]
    lines.append('EXECUTIVE SUMMARY – DB-RECHNUNG & BREAK-EVEN-ANALYSE')
    lines.append('=' * 65)
    lines.append('')
    lines.append('BASISAUSWERTUNG:')
    lines.append(f"  Umsatz:                {base['Umsatz (€)']/1e6:>8.1f} Mio. €")
    lines.append(f"  CM1:                   {base['CM1 absolut (€)']/1e6:>8.1f} Mio. €  ({base['CM1-Quote (%)']:.1f}%)")
    lines.append(f"  Break-Even-Umsatz:     {base['Break-Even-Umsatz (€)']/1e6:>8.1f} Mio. €")
    lines.append(f"  Sicherheitsspanne:     {base['Sicherheitsspanne (€)']/1e6:>8.1f} Mio. €  ({base['Sicherheitsspanne (%)']:.1f}%)")
    lines.append(f"  EBITDA:                {base['EBITDA (€)']/1e6:>8.1f} Mio. €  ({base['EBITDA-Marge (%)']:.1f}%)")
    lines.append('')
    lines.append('SZENARIOVERGLEICH:')
    for sc in scenarios:
        lines.append(f"  {sc['Szenario']:<8}: EBITDA {sc['EBITDA (€)']/1e6:>7.1f} Mio. €  "
                     f"| Break-Even {sc['Break-Even-Umsatz (€)']/1e6:.1f} Mio. €  "
                     f"| Sicherheitsspanne {sc['Sicherheitsspanne (%)']:.1f}%")
    lines.append('')
    lines.append('KRITISCHE STELLSCHRAUBEN:')

    # Identify the most sensitive lever
    worst = next(sc for sc in scenarios if sc['Szenario'] == 'Worst')
    best  = next(sc for sc in scenarios if sc['Szenario'] == 'Best')
    ebitda_range = best['EBITDA (€)'] - worst['EBITDA (€)']
    lines.append(f"  → EBITDA-Spanne Base→Best→Worst: {ebitda_range/1e6:.1f} Mio. €")
    lines.append(f"  → Umsatz {REVENUE_DELTA:.0%} & Materialkosten {MATERIAL_DELTA:.0%} sind die dominanten Treiber.")
    lines.append('')
    lines.append('─' * 65)
    lines.append('Hinweis: Schulungs-Demodaten – keine realen Unternehmensdaten.')
    lines.append('         Berechnungen immer mit Quell-Excel gegenchecken!')
    return '\n'.join(lines)


print(generate_executive_summary(base_result, scenario_results))

---

## Nächste Schritte

1. **Eigene Daten eintragen:** `REVENUE`, `VARIABLE_COSTS`, `FIXED_COSTS` in Schritt 2 anpassen
2. **5-Jahres-Analyse:** Schleife über alle Jahresspalten aus der Excel-Datei
3. **Claude fragen:** Diesen Code in Claude einfügen und z.B. fragen:
   > *"Erweitere den Code um eine rollierende 5-Jahres-Trendlinie für Break-Even und EBITDA-Marge."*
4. **Export:** `fig.write_image('chart.png')` benötigt `kaleido` (`pip install kaleido`)

---

*Notebook erstellt im Rahmen der Schulung: KI im Controlling | März 2026*  
*Basierend auf Claude-generiertem Code – immer kritisch prüfen!*